In [1]:

# ============================================================
# ZEROQ AI - NOTEBOOK 03
# RELATIONAL INTEGRITY VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")


# ------------------------------------------------------------
# 2. DEFINE DATASET FILENAMES
# ------------------------------------------------------------

DATASET_FILES = {
    "products": "products_cleaned.csv",
    "customers": "customers_cleaned.csv",
    "facilities": "facilities_cleaned.csv",
    "production_lines": "production_lines_cleaned.csv",
    "materials": "materials_cleaned.csv",
    "production_batches": "production_batches_cleaned.csv",
    "complaints": "complaints_cleaned.csv",
    "quality_inspections": "quality_inspections_cleaned.csv",
    "investigations": "investigations_cleaned.csv",
    "corrective_actions": "corrective_actions_cleaned.csv",
    "data_dictionary": "data_dictionary_cleaned.csv"
}

REQUIRED_FILES = set(DATASET_FILES.values())


# ------------------------------------------------------------
# 3. AUTOMATICALLY LOCATE PROCESSED DATA FOLDER
# ------------------------------------------------------------

print("\nSearching for cleaned datasets...")

search_roots = [Path.cwd(), *Path.cwd().parents]

PROCESSED_DATA_PATH = None

for root in search_roots:

    for file_path in root.rglob("products_cleaned.csv"):

        candidate_folder = file_path.parent

        available_files = {
            file.name for file in candidate_folder.glob("*.csv")
        }

        if REQUIRED_FILES.issubset(available_files):
            PROCESSED_DATA_PATH = candidate_folder
            break

    if PROCESSED_DATA_PATH is not None:
        break

if PROCESSED_DATA_PATH is None:

    raise FileNotFoundError(
        "Could not locate the folder containing all 11 cleaned CSV files. "
        "Execute Notebook 2 first and verify the processed_data folder."
    )

PROJECT_PATH = PROCESSED_DATA_PATH.parent

PREPROCESSING_PATH = PROJECT_PATH / "preprocessing"
REPORT_PATH = PREPROCESSING_PATH / "reports"

REPORT_PATH.mkdir(parents=True, exist_ok=True)

print("\nPROJECT CONFIGURATION")
print("-" * 65)
print("Project directory:", PROJECT_PATH)
print("Processed data directory:", PROCESSED_DATA_PATH)
print("Reports directory:", REPORT_PATH)


# ------------------------------------------------------------
# 4. LOAD ALL CLEANED DATASETS
# ------------------------------------------------------------

datasets = {}

for dataset_name, filename in DATASET_FILES.items():

    file_path = PROCESSED_DATA_PATH / filename

    # Read identifiers as strings to avoid type mismatches
    # during primary key and foreign key comparisons.
    df = pd.read_csv(
        file_path,
        dtype=str,
        keep_default_na=True
    )

    # Standardize whitespace in column names and text values
    df.columns = df.columns.str.strip()

    for column in df.select_dtypes(include=["object"]).columns:

        df[column] = df[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

        df[column] = df[column].replace(
            r"^\s*$",
            np.nan,
            regex=True
        )

    datasets[dataset_name] = df

    print(
        f"{dataset_name:25} | "
        f"Loaded | "
        f"Shape: {df.shape}"
    )

print("\nTotal datasets loaded:", len(datasets))

if len(datasets) != 11:
    raise ValueError("Expected 11 cleaned datasets.")


# ------------------------------------------------------------
# 5. DEFINE PRIMARY KEY RELATIONSHIPS
# ------------------------------------------------------------

PRIMARY_KEYS = {
    "products": "product_id",
    "customers": "customer_id",
    "facilities": "facility_id",
    "production_lines": "production_line_id",
    "materials": "material_id",
    "production_batches": "batch_id",
    "complaints": "complaint_id",
    "quality_inspections": "inspection_id",
    "investigations": "investigation_id",
    "corrective_actions": "action_id"
}


# ------------------------------------------------------------
# 6. DEFINE FOREIGN KEY RELATIONSHIPS
# ------------------------------------------------------------

# Format:
# (Child Dataset, Foreign Key Column,
#  Parent Dataset, Parent Primary Key)

FOREIGN_KEY_RELATIONSHIPS = [

    # Production batch relationships
    (
        "production_batches",
        "product_id",
        "products",
        "product_id"
    ),

    (
        "production_batches",
        "facility_id",
        "facilities",
        "facility_id"
    ),

    (
        "production_batches",
        "production_line_id",
        "production_lines",
        "production_line_id"
    ),

    (
        "production_batches",
        "packaging_material_id",
        "materials",
        "material_id"
    ),

    (
        "production_batches",
        "raw_material_id",
        "materials",
        "material_id"
    ),

    # Complaint relationships
    (
        "complaints",
        "customer_id",
        "customers",
        "customer_id"
    ),

    (
        "complaints",
        "product_id",
        "products",
        "product_id"
    ),

    (
        "complaints",
        "batch_id",
        "production_batches",
        "batch_id"
    ),

    # Quality inspection relationships
    (
        "quality_inspections",
        "batch_id",
        "production_batches",
        "batch_id"
    ),

    # Investigation relationships
    (
        "investigations",
        "complaint_id",
        "complaints",
        "complaint_id"
    ),

    # Corrective action relationships
    (
        "corrective_actions",
        "investigation_id",
        "investigations",
        "investigation_id"
    ),

    (
        "corrective_actions",
        "complaint_id",
        "complaints",
        "complaint_id"
    )
]

print("\nTotal foreign key relationships to validate:",
      len(FOREIGN_KEY_RELATIONSHIPS))


# ------------------------------------------------------------
# 7. PRIMARY KEY VALIDATION
# ------------------------------------------------------------

primary_key_reports = []

for dataset_name, primary_key in PRIMARY_KEYS.items():

    df = datasets[dataset_name]

    if primary_key not in df.columns:

        primary_key_reports.append({
            "Dataset": dataset_name,
            "Primary_Key": primary_key,
            "Total_Rows": len(df),
            "Null_Keys": None,
            "Duplicate_Keys": None,
            "Unique_Keys": None,
            "Status": "COLUMN NOT FOUND"
        })

        continue

    null_count = int(df[primary_key].isna().sum())

    duplicate_count = int(
        df.loc[df[primary_key].notna(), primary_key].duplicated().sum()
    )

    unique_count = int(df[primary_key].nunique(dropna=True))

    status = (
        "PASS"
        if null_count == 0 and duplicate_count == 0
        else "CHECK REQUIRED"
    )

    primary_key_reports.append({
        "Dataset": dataset_name,
        "Primary_Key": primary_key,
        "Total_Rows": len(df),
        "Null_Keys": null_count,
        "Duplicate_Keys": duplicate_count,
        "Unique_Keys": unique_count,
        "Status": status
    })

primary_key_report_df = pd.DataFrame(primary_key_reports)

print("\n" + "=" * 75)
print("PRIMARY KEY VALIDATION")
print("=" * 75)

display(primary_key_report_df)


# ------------------------------------------------------------
# 8. FOREIGN KEY VALIDATION
# ------------------------------------------------------------

foreign_key_reports = []
orphan_records = []

for (
    child_dataset,
    foreign_key,
    parent_dataset,
    parent_primary_key
) in FOREIGN_KEY_RELATIONSHIPS:

    child_df = datasets[child_dataset]
    parent_df = datasets[parent_dataset]

    print(
        f"\nValidating: {child_dataset}.{foreign_key}"
        f" -> {parent_dataset}.{parent_primary_key}"
    )

    # Check whether the required columns exist
    if (
        foreign_key not in child_df.columns
        or parent_primary_key not in parent_df.columns
    ):

        foreign_key_reports.append({
            "Child_Dataset": child_dataset,
            "Foreign_Key": foreign_key,
            "Parent_Dataset": parent_dataset,
            "Parent_Primary_Key": parent_primary_key,
            "Child_Rows": len(child_df),
            "Null_Foreign_Keys": None,
            "Non_Null_Foreign_Keys": None,
            "Matching_Rows": None,
            "Orphan_Rows": None,
            "Match_Percentage": None,
            "Status": "COLUMN NOT FOUND"
        })

        continue

    # Get child foreign key values
    child_values = child_df[foreign_key]

    # Get valid parent primary key values
    parent_values = set(
        parent_df[parent_primary_key].dropna()
    )

    # Identify missing foreign keys
    null_count = int(child_values.isna().sum())

    non_null_values = child_values.dropna()

    # Identify values that do not exist in parent dataset
    orphan_mask = (
        child_values.notna()
        & ~child_values.isin(parent_values)
    )

    orphan_count = int(orphan_mask.sum())

    matching_count = int(
        child_values.notna().sum() - orphan_count
    )

    non_null_count = len(non_null_values)

    match_percentage = (
        matching_count / non_null_count * 100
        if non_null_count > 0 else 0
    )

    # Save orphan records for investigation
    if orphan_count > 0:

        orphan_df = child_df.loc[
            orphan_mask
        ].copy()

        orphan_df["Source_Dataset"] = child_dataset
        orphan_df["Foreign_Key_Column"] = foreign_key
        orphan_df["Invalid_Foreign_Key_Value"] = (
            child_df.loc[orphan_mask, foreign_key].values
        )

        orphan_records.append(orphan_df)

    # Validation status
    if orphan_count == 0:
        status = "PASS"
    else:
        status = "ORPHAN RECORDS FOUND"

    foreign_key_reports.append({

        "Child_Dataset": child_dataset,
        "Foreign_Key": foreign_key,
        "Parent_Dataset": parent_dataset,
        "Parent_Primary_Key": parent_primary_key,
        "Child_Rows": len(child_df),
        "Null_Foreign_Keys": null_count,
        "Non_Null_Foreign_Keys": non_null_count,
        "Matching_Rows": matching_count,
        "Orphan_Rows": orphan_count,
        "Match_Percentage": round(match_percentage, 2),
        "Status": status
    })

    print("Null foreign keys:", null_count)
    print("Matching records:", matching_count)
    print("Orphan records:", orphan_count)
    print("Match percentage:", round(match_percentage, 2), "%")


foreign_key_report_df = pd.DataFrame(
    foreign_key_reports
)

print("\n" + "=" * 75)
print("FOREIGN KEY VALIDATION REPORT")
print("=" * 75)

display(foreign_key_report_df)


# ------------------------------------------------------------
# 9. COLLECT ORPHAN RECORD DETAILS
# ------------------------------------------------------------

if orphan_records:

    orphan_records_df = pd.concat(
        orphan_records,
        ignore_index=True
    )

else:

    orphan_records_df = pd.DataFrame(
        columns=[
            "Source_Dataset",
            "Foreign_Key_Column",
            "Invalid_Foreign_Key_Value"
        ]
    )

print("\nORPHAN RECORD SUMMARY")
print("-" * 65)
print("Total orphan records:", len(orphan_records_df))

if not orphan_records_df.empty:
    display(orphan_records_df.head(20))
else:
    print("No orphan records detected.")


# ------------------------------------------------------------
# 10. RELATIONSHIP CARDINALITY ANALYSIS
# ------------------------------------------------------------

cardinality_reports = []

for (
    child_dataset,
    foreign_key,
    parent_dataset,
    parent_primary_key
) in FOREIGN_KEY_RELATIONSHIPS:

    child_df = datasets[child_dataset]
    parent_df = datasets[parent_dataset]

    if (
        foreign_key not in child_df.columns
        or parent_primary_key not in parent_df.columns
    ):
        continue

    child_values = child_df[foreign_key].dropna()

    parent_values = set(
        parent_df[parent_primary_key].dropna()
    )

    valid_child_values = child_values[
        child_values.isin(parent_values)
    ]

    parent_keys_referenced = valid_child_values.nunique()

    max_child_records_per_parent = (
        int(valid_child_values.value_counts().max())
        if not valid_child_values.empty
        else 0
    )

    cardinality_reports.append({

        "Child_Dataset": child_dataset,
        "Foreign_Key": foreign_key,
        "Parent_Dataset": parent_dataset,
        "Parent_Primary_Key": parent_primary_key,

        "Child_Rows_With_Valid_Reference": len(
            valid_child_values
        ),

        "Distinct_Parent_Records_Referenced": (
            parent_keys_referenced
        ),

        "Maximum_Child_Records_Per_Parent": (
            max_child_records_per_parent
        ),

        "Relationship_Type": "Many-to-One"
    })


cardinality_report_df = pd.DataFrame(
    cardinality_reports
)

print("\n" + "=" * 75)
print("RELATIONSHIP CARDINALITY ANALYSIS")
print("=" * 75)

display(cardinality_report_df)


# ------------------------------------------------------------
# 11. CHECK UNREFERENCED PARENT RECORDS
# ------------------------------------------------------------

# Parent records without any matching child record are not
# necessarily errors. This is an informational analysis.

unreferenced_parent_reports = []

for (
    child_dataset,
    foreign_key,
    parent_dataset,
    parent_primary_key
) in FOREIGN_KEY_RELATIONSHIPS:

    child_df = datasets[child_dataset]
    parent_df = datasets[parent_dataset]

    if (
        foreign_key not in child_df.columns
        or parent_primary_key not in parent_df.columns
    ):
        continue

    referenced_values = set(
        child_df[foreign_key].dropna()
    )

    parent_keys = parent_df[parent_primary_key]

    unreferenced_mask = (
        parent_keys.notna()
        & ~parent_keys.isin(referenced_values)
    )

    unreferenced_count = int(unreferenced_mask.sum())

    unreferenced_parent_reports.append({

        "Parent_Dataset": parent_dataset,
        "Parent_Primary_Key": parent_primary_key,
        "Child_Dataset": child_dataset,
        "Foreign_Key": foreign_key,
        "Unreferenced_Parent_Records": unreferenced_count,
        "Status": (
            "UNREFERENCED RECORDS FOUND"
            if unreferenced_count > 0
            else "ALL REFERENCED"
        )
    })


unreferenced_parent_report_df = pd.DataFrame(
    unreferenced_parent_reports
)

print("\nUNREFERENCED PARENT RECORD ANALYSIS")
display(unreferenced_parent_report_df)


# ------------------------------------------------------------
# 12. SAVE ALL RELATIONAL INTEGRITY REPORTS
# ------------------------------------------------------------

RELATIONAL_REPORT_FILE = (
    REPORT_PATH / "relational_integrity_report.xlsx"
)

with pd.ExcelWriter(
    RELATIONAL_REPORT_FILE,
    engine="openpyxl"
) as writer:

    primary_key_report_df.to_excel(
        writer,
        sheet_name="Primary_Key_Audit",
        index=False
    )

    foreign_key_report_df.to_excel(
        writer,
        sheet_name="Foreign_Key_Audit",
        index=False
    )

    orphan_records_df.to_excel(
        writer,
        sheet_name="Orphan_Records",
        index=False
    )

    cardinality_report_df.to_excel(
        writer,
        sheet_name="Cardinality_Analysis",
        index=False
    )

    unreferenced_parent_report_df.to_excel(
        writer,
        sheet_name="Unreferenced_Parents",
        index=False
    )

print("\nRelational integrity report saved:")
print(RELATIONAL_REPORT_FILE)


# ------------------------------------------------------------
# 13. FINAL RELATIONAL INTEGRITY SUMMARY
# ------------------------------------------------------------

primary_key_issues = (
    len(
        primary_key_report_df[
            primary_key_report_df["Status"] != "PASS"
        ]
    )
)

foreign_key_issues = (
    len(
        foreign_key_report_df[
            foreign_key_report_df["Status"] != "PASS"
        ]
    )
)

total_orphans = (
    int(foreign_key_report_df["Orphan_Rows"].fillna(0).sum())
    if not foreign_key_report_df.empty
    else 0
)

total_missing_foreign_keys = (
    int(
        foreign_key_report_df[
            "Null_Foreign_Keys"
        ].fillna(0).sum()
    )
    if not foreign_key_report_df.empty
    else 0
)

print("\n")
print("=" * 75)
print("ZEROQ AI - RELATIONAL INTEGRITY VALIDATION SUMMARY")
print("=" * 75)

print("Total datasets loaded:", len(datasets))

print(
    "Primary key relationships checked:",
    len(primary_key_report_df)
)

print(
    "Foreign key relationships checked:",
    len(foreign_key_report_df)
)

print(
    "Primary key issues:",
    primary_key_issues
)

print(
    "Foreign key relationships requiring attention:",
    foreign_key_issues
)

print(
    "Total orphan records:",
    total_orphans
)

print(
    "Total missing foreign key values:",
    total_missing_foreign_keys
)

print("\nReport location:")
print(RELATIONAL_REPORT_FILE)

print("-" * 75)

if primary_key_issues == 0 and foreign_key_issues == 0:

    print("RELATIONAL INTEGRITY STATUS: PASSED")

else:

    print("RELATIONAL INTEGRITY STATUS: REVIEW REQUIRED")

print("=" * 75)

print("\nIMPORTANT:")
print("No records were deleted or modified.")
print("All detected issues are preserved in the audit report.")


Libraries imported successfully.

Searching for cleaned datasets...

PROJECT CONFIGURATION
-----------------------------------------------------------------
Project directory: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training
Processed data directory: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\processed_data
Reports directory: d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing\reports
products                  | Loaded | Shape: (20, 6)
customers                 | Loaded | Shape: (500, 5)
facilities                | Loaded | Shape: (5, 4)
production_lines          | Loaded | Shape: (15, 6)
materials                 | Loaded | Shape: (40, 6)
production_batches        | Loaded | Shape: (24000, 20)
complaints                | Loaded | Shape: (6000, 14)
quality_inspections       | Loaded | Shape: (37245, 11)
investigations            | Loaded | Shape: (4000, 10)
corrective_actions        | Loaded | Shape: (3500, 13)
data_dictionary           | Loaded | 

,Dataset,Primary_Key,Total_Rows,Null_Keys,Duplicate_Keys,Unique_Keys,Status
0,products,product_id,20,0,0,20,PASS
1,customers,customer_id,500,0,0,500,PASS
2,facilities,facility_id,5,0,0,5,PASS
3,production_lines,production_line_id,15,0,0,15,PASS
4,materials,material_id,40,0,0,40,PASS
5,production_batches,batch_id,24000,0,0,24000,PASS
6,complaints,complaint_id,6000,0,0,6000,PASS
7,quality_inspections,inspection_id,37245,0,0,37245,PASS
8,investigations,investigation_id,4000,0,0,4000,PASS
9,corrective_actions,action_id,3500,0,0,3500,PASS



Validating: production_batches.product_id -> products.product_id
Null foreign keys: 0
Matching records: 24000
Orphan records: 0
Match percentage: 100.0 %

Validating: production_batches.facility_id -> facilities.facility_id
Null foreign keys: 0
Matching records: 24000
Orphan records: 0
Match percentage: 100.0 %

Validating: production_batches.production_line_id -> production_lines.production_line_id
Null foreign keys: 0
Matching records: 24000
Orphan records: 0
Match percentage: 100.0 %

Validating: production_batches.packaging_material_id -> materials.material_id
Null foreign keys: 0
Matching records: 24000
Orphan records: 0
Match percentage: 100.0 %

Validating: production_batches.raw_material_id -> materials.material_id
Null foreign keys: 0
Matching records: 24000
Orphan records: 0
Match percentage: 100.0 %

Validating: complaints.customer_id -> customers.customer_id
Null foreign keys: 0
Matching records: 6000
Orphan records: 0
Match percentage: 100.0 %

Validating: complaints.prod

,Child_Dataset,Foreign_Key,Parent_Dataset,Parent_Primary_Key,Child_Rows,Null_Foreign_Keys,Non_Null_Foreign_Keys,Matching_Rows,Orphan_Rows,Match_Percentage,Status
0,production_batches,product_id,products,product_id,24000,0,24000,24000,0,100.0,PASS
1,production_batches,facility_id,facilities,facility_id,24000,0,24000,24000,0,100.0,PASS
2,production_batches,production_line_id,production_lines,production_line_id,24000,0,24000,24000,0,100.0,PASS
3,production_batches,packaging_material_id,materials,material_id,24000,0,24000,24000,0,100.0,PASS
4,production_batches,raw_material_id,materials,material_id,24000,0,24000,24000,0,100.0,PASS
5,complaints,customer_id,customers,customer_id,6000,0,6000,6000,0,100.0,PASS
6,complaints,product_id,products,product_id,6000,0,6000,6000,0,100.0,PASS
7,complaints,batch_id,production_batches,batch_id,6000,0,6000,6000,0,100.0,PASS
8,quality_inspections,batch_id,production_batches,batch_id,37245,0,37245,37245,0,100.0,PASS
9,investigations,complaint_id,complaints,complaint_id,4000,0,4000,4000,0,100.0,PASS



ORPHAN RECORD SUMMARY
-----------------------------------------------------------------
Total orphan records: 0
No orphan records detected.

RELATIONSHIP CARDINALITY ANALYSIS


,Child_Dataset,Foreign_Key,Parent_Dataset,Parent_Primary_Key,Child_Rows_With_Valid_Reference,Distinct_Parent_Records_Referenced,Maximum_Child_Records_Per_Parent,Relationship_Type
0,production_batches,product_id,products,product_id,24000,20,1277,Many-to-One
1,production_batches,facility_id,facilities,facility_id,24000,5,4873,Many-to-One
2,production_batches,production_line_id,production_lines,production_line_id,24000,15,1752,Many-to-One
3,production_batches,packaging_material_id,materials,material_id,24000,8,3132,Many-to-One
4,production_batches,raw_material_id,materials,material_id,24000,32,808,Many-to-One
5,complaints,customer_id,customers,customer_id,6000,500,25,Many-to-One
6,complaints,product_id,products,product_id,6000,20,341,Many-to-One
7,complaints,batch_id,production_batches,batch_id,6000,5314,4,Many-to-One
8,quality_inspections,batch_id,production_batches,batch_id,37245,24000,3,Many-to-One
9,investigations,complaint_id,complaints,complaint_id,4000,4000,1,Many-to-One



UNREFERENCED PARENT RECORD ANALYSIS


,Parent_Dataset,Parent_Primary_Key,Child_Dataset,Foreign_Key,Unreferenced_Parent_Records,Status
0,products,product_id,production_batches,product_id,0,ALL REFERENCED
1,facilities,facility_id,production_batches,facility_id,0,ALL REFERENCED
2,production_lines,production_line_id,production_batches,production_line_id,0,ALL REFERENCED
3,materials,material_id,production_batches,packaging_material_id,32,UNREFERENCED RECORDS FOUND
4,materials,material_id,production_batches,raw_material_id,8,UNREFERENCED RECORDS FOUND
5,customers,customer_id,complaints,customer_id,0,ALL REFERENCED
6,products,product_id,complaints,product_id,0,ALL REFERENCED
7,production_batches,batch_id,complaints,batch_id,18686,UNREFERENCED RECORDS FOUND
8,production_batches,batch_id,quality_inspections,batch_id,0,ALL REFERENCED
9,complaints,complaint_id,investigations,complaint_id,2000,UNREFERENCED RECORDS FOUND



Relational integrity report saved:
d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing\reports\relational_integrity_report.xlsx


ZEROQ AI - RELATIONAL INTEGRITY VALIDATION SUMMARY
Total datasets loaded: 11
Primary key relationships checked: 10
Foreign key relationships checked: 12
Primary key issues: 0
Foreign key relationships requiring attention: 0
Total orphan records: 0
Total missing foreign key values: 0

Report location:
d:\AI WORKSHOP\Hackathon\Sakthi Hackathon - ML Training\preprocessing\reports\relational_integrity_report.xlsx
---------------------------------------------------------------------------
RELATIONAL INTEGRITY STATUS: PASSED

IMPORTANT:
No records were deleted or modified.
All detected issues are preserved in the audit report.


In [ ]:
print("HI")